In [3]:
import os
import numpy as np
import pandas as pd
import folium
import branca
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
import seaborn as sns
from datetime import datetime
import math
import glob

# Configuration Parameters
RUNTIME_CONFIG = {
    'DATA_FILES_PATTERN': 'JC2024*citibiketripdata.csv',  
    'DATA_DIR': 'data/citibike',  
    'OUTPUT_DIR': 'results',  
    'MODE': 'visualize',  
    'MAX_STATIONS': 50,  
    'SAMPLE_RATE': 1, 
    'DISTANCE_THRESHOLD': 2.0,  
}

# Citibike time period configurations
TIME_PERIODS = {
    "Morning Peak (7-10 AM)": (7, 10),
    "Afternoon (10 AM-4 PM)": (10, 16),
    "Evening Peak (4-7 PM)": (16, 19),
    "All Day": (0, 24)
}

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate haversine distance between two points in kilometers"""
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371  # Radius of earth in kilometers
    
    return c * r

def create_bezier_curve(lon1, lat1, lon2, lat2, num_points=20):
    """Create smooth bezier curve between two points"""
    # Calculate distance for dynamic curve strength
    dx = lon2 - lon1
    dy = lat2 - lat1
    distance = math.sqrt(dx**2 + dy**2)
    
    # Calculate midpoint
    mid_lon = (lon1 + lon2) / 2
    mid_lat = (lat1 + lat2) / 2
    
    # Dynamic curve strength based on distance
    curve_strength = min(0.3, 0.05 + 0.1 * (distance / 0.05))
    
    # Calculate control point perpendicular to the line
    control_lon = mid_lon - dy * curve_strength
    control_lat = mid_lat + dx * curve_strength
    
    # Generate curve points
    t = np.linspace(0, 1, num_points)
    curve = []
    for ti in t:
        x = (1-ti)**2 * lon1 + 2*(1-ti)*ti * control_lon + ti**2 * lon2
        y = (1-ti)**2 * lat1 + 2*(1-ti)*ti * control_lat + ti**2 * lat2
        curve.append((x, y))
    
    return curve

def load_citibike_data(data_files_pattern, sample_rate=1.0):
    """Load and preprocess Citibike data from multiple files"""
    print(f"Loading Citibike data with {sample_rate*100:.1f}% sampling rate")
    
    # Find all matching files
    data_dir = os.path.dirname(data_files_pattern)
    if not data_dir:
        data_dir = '.'
    file_list = glob.glob(os.path.join(data_dir, os.path.basename(data_files_pattern)))
    
    if not file_list:
        raise FileNotFoundError(f"No data files found matching pattern: {data_files_pattern}")
    
    file_list.sort()  # Ensure files are processed in order
    print(f"Found {len(file_list)} files to process")
    
    # Load data from each file
    all_trips = []
    
    for file_path in file_list:
        file_name = os.path.basename(file_path)
        print(f"Processing file: {file_name}")
        
        # Extract month from filename (assuming format JC2024MMcitibiketripdata.csv)
        try:
            month = int(file_name[6:8])
            print(f"Extracted month: {month}")
        except (ValueError, IndexError):
            month = 0
            print(f"Could not extract month from filename: {file_name}, using placeholder")
        
        # Load data in chunks to save memory
        chunks = []
        
        for chunk in pd.read_csv(file_path, chunksize=10000):
            # Sample the chunk to reduce memory usage
            sampled_chunk = chunk.sample(frac=sample_rate, random_state=42)
            chunks.append(sampled_chunk)
        
        # Combine chunks for this month
        if chunks:
            monthly_trips = pd.concat(chunks)
            print(f"Loaded {len(monthly_trips)} sampled trips from {file_name}")
            
            # Add month column if not already present
            if 'month' not in monthly_trips.columns:
                monthly_trips['month'] = month
            
            all_trips.append(monthly_trips)
        else:
            print(f"No data loaded from {file_name}")
    
    # Combine all monthly data
    if not all_trips:
        raise ValueError("No trip data was loaded from any file")
    
    trips_df = pd.concat(all_trips)
    print(f"Combined dataset contains {len(trips_df)} trips")
    
    # Basic data cleaning
    trips_df = trips_df.dropna(subset=['start_station_name', 'end_station_name', 
                                     'start_lat', 'start_lng', 
                                     'end_lat', 'end_lng'])
    
    # Convert timestamps to datetime
    print("Converting timestamps to datetime...")
    trips_df['started_at'] = pd.to_datetime(trips_df['started_at'], errors='coerce')
    trips_df['ended_at'] = pd.to_datetime(trips_df['ended_at'], errors='coerce')
    
    # Extract time features
    trips_df['start_hour'] = trips_df['started_at'].dt.hour
    trips_df['start_day'] = trips_df['started_at'].dt.dayofweek  # Monday=0, Sunday=6
    trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Make sure month is available
    if 'month' not in trips_df.columns:
        trips_df['month'] = trips_df['started_at'].dt.month
    
    # Calculate trip duration in minutes
    trips_df['duration_minutes'] = (trips_df['ended_at'] - trips_df['started_at']).dt.total_seconds() / 60
    
    # Filter out unrealistic trips
    trips_df = trips_df[(trips_df['duration_minutes'] > 0) & 
                      (trips_df['duration_minutes'] < 24*60)]  # Less than 24 hours
    
    print(f"After cleaning: {len(trips_df)} trips")
    
    return trips_df

def get_top_stations(trips_df, max_stations):
    """Get the top stations by trip count and create station mapping"""
    # Get unique stations
    start_counts = trips_df['start_station_name'].value_counts()
    end_counts = trips_df['end_station_name'].value_counts()
    combined_counts = start_counts.add(end_counts, fill_value=0)
    
    # Take the top stations
    max_stations = min(max_stations, len(combined_counts))
    top_stations = combined_counts.sort_values(ascending=False).head(max_stations).index.tolist()
    
    print(f"Using top {len(top_stations)} stations")
    
    # Create station mapping
    station_mapping = {station: idx for idx, station in enumerate(top_stations)}
    
    # Create station data dictionary
    station_data = {}
    
    for station in top_stations:
        # Get station coordinates from the first occurrence
        start_rows = trips_df[trips_df['start_station_name'] == station]
        end_rows = trips_df[trips_df['end_station_name'] == station]
        
        if not start_rows.empty:
            lat = start_rows['start_lat'].iloc[0]
            lng = start_rows['start_lng'].iloc[0]
        elif not end_rows.empty:
            lat = end_rows['end_lat'].iloc[0]
            lng = end_rows['end_lng'].iloc[0]
        else:
            lat, lng = 0, 0
            print(f"No coordinate data found for station: {station}")
        
        # Calculate station popularity
        popularity = (
            trips_df['start_station_name'].value_counts().get(station, 0) +
            trips_df['end_station_name'].value_counts().get(station, 0)
        )
        
        station_data[station_mapping[station]] = {
            'name': station,
            'lat': lat,
            'lng': lng,
            'popularity': popularity
        }
    
    # Create a dataframe from station_data for easy access
    stations_df = pd.DataFrame([
        {
            'name': data['name'],
            'lat': data['lat'],
            'lng': data['lng'],
            'popularity': data['popularity']
        }
        for idx, data in station_data.items()
    ])
    
    return station_mapping, station_data, top_stations, stations_df

def create_interactive_station_map(stations_df, map_title="Bike Stations", output_dir="results"):
    """Create an interactive map of bike stations with multiple layer options and controls"""
    # Calculate map center
    center_lat = stations_df['lat'].mean()
    center_lng = stations_df['lng'].mean()
    
    # Create base map with multiple tile options
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=13,
        tiles=None,  # No default tiles, we'll add them as layers
        control_scale=True
    )
    
    # Add multiple tile layers
    folium.TileLayer('cartodbpositron', name='CartoDB Positron', control=True).add_to(m)
    folium.TileLayer('openstreetmap', name='OpenStreetMap', control=True).add_to(m)
    
    # Create feature groups for better organization
    stations_group = folium.FeatureGroup(name="All Stations", show=True)
    popular_group = folium.FeatureGroup(name="Popular Stations", show=False)
    clusters_group = MarkerCluster(name="Clustered Stations")
    
    # Find threshold for popular stations (top 25%)
    popularity_threshold = stations_df['popularity'].quantile(0.75)
    
    # Create a colormap for station markers based on popularity
    max_popularity = stations_df['popularity'].max()
    colormap = branca.colormap.LinearColormap(
        colors=['green', 'yellow', 'orange', 'red'],
        vmin=0,
        vmax=max_popularity,
        caption='Station Popularity'
    )
    
    # Add station markers
    for _, station in stations_df.iterrows():
        # Create popup with station information
        popup_html = f"""
        <div style="font-family: Arial; width: 200px;">
            <h3 style="margin-top: 0;">{station['name']}</h3>
            <p><b>Popularity:</b> {station['popularity']}</p>
            <p><b>Location:</b> {station['lat']:.4f}, {station['lng']:.4f}</p>
        </div>
        """
        popup = folium.Popup(popup_html, max_width=300)
        
        # Determine color based on popularity
        color = colormap(station['popularity'])
        
        # Create circle marker
        marker = folium.CircleMarker(
            location=[station['lat'], station['lng']],
            radius=6,
            color='black',
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            popup=popup,
            tooltip=station['name']
        )
        
        # Add marker to appropriate groups
        marker.add_to(stations_group)
        
        # Also add to popular group if it's a popular station
        if station['popularity'] >= popularity_threshold:
            folium.CircleMarker(
                location=[station['lat'], station['lng']],
                radius=8,
                color='black',
                weight=2,
                fill=True,
                fill_color=color,
                fill_opacity=0.9,
                popup=popup,
                tooltip=f"{station['name']} (Popular)"
            ).add_to(popular_group)
        
        # Add to cluster group as well
        folium.Marker(
            location=[station['lat'], station['lng']],
            popup=popup,
            tooltip=station['name'],
            icon=folium.Icon(color='blue', icon='bicycle', prefix='fa')
        ).add_to(clusters_group)
    
    # Add all feature groups to the map
    stations_group.add_to(m)
    popular_group.add_to(m)
    clusters_group.add_to(m)
    
    # Add colormap to the map
    colormap.add_to(m)
    
    # Add a title using HTML
    title_html = f'''
        <div style="position: fixed; 
                    top: 10px; left: 50px; width: 300px; height: 40px; 
                    background-color: white; border-radius: 5px;
                    z-index: 9999; font-size: 16px; font-weight: bold;
                    text-align: center; line-height: 40px;
                    box-shadow: 0 0 5px rgba(0,0,0,0.4);">
            {map_title}
        </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    # Add a draggable legend with custom CSS and JavaScript
    legend_html = '''
    <div id="legend" style="position: fixed; bottom: 50px; right: 50px; 
                  z-index: 9999; background-color: white; padding: 10px;
                  border-radius: 5px; box-shadow: 0 0 5px rgba(0,0,0,0.4);">
      <h4 style="margin-top: 0;">Legend</h4>
      <div>
        <i style="background: green; border-radius: 50%; width: 10px; height: 10px; display: inline-block;"></i>
        <span>Low popularity</span>
      </div>
      <div>
        <i style="background: yellow; border-radius: 50%; width: 10px; height: 10px; display: inline-block;"></i>
        <span>Medium popularity</span>
      </div>
      <div>
        <i style="background: red; border-radius: 50%; width: 10px; height: 10px; display: inline-block;"></i>
        <span>High popularity</span>
      </div>
      <div style="font-size: 10px; margin-top: 8px; color: #666; font-style: italic;">
        Drag me by clicking and holding
      </div>
    </div>
    
    <script>
      // Make the legend draggable
      var legend = document.getElementById('legend');
      var isDragging = false;
      var offsetX, offsetY;
      
      legend.addEventListener('mousedown', function(e) {
        isDragging = true;
        offsetX = e.clientX - legend.getBoundingClientRect().left;
        offsetY = e.clientY - legend.getBoundingClientRect().top;
        legend.style.cursor = 'grabbing';
      });
      
      document.addEventListener('mousemove', function(e) {
        if (isDragging) {
          legend.style.left = (e.clientX - offsetX) + 'px';
          legend.style.top = (e.clientY - offsetY) + 'px';
          legend.style.right = 'auto';
          legend.style.bottom = 'auto';
        }
      });
      
      document.addEventListener('mouseup', function() {
        isDragging = false;
        legend.style.cursor = 'grab';
      });
      
      legend.style.cursor = 'grab';
    </script>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # Add layer control directly from folium (not plugins)
    folium.LayerControl(position='topright', collapsed=False).add_to(m)
    
    # Save map
    os.makedirs(output_dir, exist_ok=True)
    map_path = os.path.join(output_dir, "interactive_station_map.html")
    m.save(map_path)
    print(f"Interactive station map saved to {map_path}")
    
    return m

def create_flow_map(stations_df, flows, output_dir="results", title="Bike Trip Flows"):
    """Create interactive flow map with animated Bezier curves"""
    if not flows:
        print("No flows to visualize")
        return None
    
    # Calculate max flow for colormap scaling
    max_flow = max(count for _, _, count in flows)
    
    # Create a custom colormap
    colormap = branca.colormap.LinearColormap(
        colors=['#1a9850', '#91cf60', '#d9ef8b', '#fee08b', '#fc8d59', '#d73027'],
        vmin=0,
        vmax=max_flow,
        caption='Number of Trips'
    )
    
    # Calculate map center from stations
    center_lat = stations_df['lat'].mean()
    center_lng = stations_df['lng'].mean()
    
    # Base map with multiple tile options
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=13,
        tiles=None,
        control_scale=True
    )
    
    # Add tile layers
    folium.TileLayer('cartodbpositron', name='CartoDB Positron', control=True).add_to(m)
    folium.TileLayer('openstreetmap', name='OpenStreetMap', control=True).add_to(m)
    
    # Create feature groups
    station_layer = folium.FeatureGroup(name='Stations', show=True)
    flow_layer = folium.FeatureGroup(name='Trip Flows', show=True)
    heatmap_layer = folium.FeatureGroup(name='Station Popularity', show=False)
    
    # Add stations to the map
    for _, station in stations_df.iterrows():
        # Create popup
        popup = folium.Popup(
            f"<b>{station['name']}</b><br>Popularity: {station['popularity']}",
            max_width=300
        )
        
        # Add marker to station layer
        folium.CircleMarker(
            location=[station['lat'], station['lng']],
            radius=5,
            color='#3186cc',
            fill=True,
            fill_color='#3186cc',
            fill_opacity=0.7,
            popup=popup,
            tooltip=station['name']
        ).add_to(station_layer)
    
    # Add flow lines with bezier curves
    for origin, destination, count in flows:
        try:
            # Get station coordinates
            orig = stations_df[stations_df['name'] == origin].iloc[0]
            dest = stations_df[stations_df['name'] == destination].iloc[0]
            
            # Skip if origin and destination are the same
            if origin == destination:
                continue
            
            # Create bezier curve for flow line
            curve_points = create_bezier_curve(
                orig['lng'], orig['lat'],
                dest['lng'], dest['lat']
            )
            
            # Calculate line weight based on flow count (normalized)
            weight = 1 + 5 * (count / max_flow)
            
            # Add line to flow layer
            folium.PolyLine(
                locations=[[lat, lng] for lng, lat in curve_points],
                color=colormap(count),
                weight=weight,
                opacity=0.8,
                tooltip=f"{origin} → {destination}: {count} trips"
            ).add_to(flow_layer)
        except (IndexError, KeyError):
            print(f"Could not find station coordinates for {origin} or {destination}")
    
    # Add all layers to map
    station_layer.add_to(m)
    flow_layer.add_to(m)
    
    # Add colormap to map
    colormap.add_to(m)
    
    # Add title
    title_html = f'''
        <div style="position: fixed; 
                    top: 10px; left: 50px; width: 300px; height: 40px; 
                    background-color: white; border-radius: 5px;
                    z-index: 9999; font-size: 16px; font-weight: bold;
                    text-align: center; line-height: 40px;
                    box-shadow: 0 0 5px rgba(0,0,0,0.4);">
            {title}
        </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    # Add draggable legend
    legend_html = '''
    <div id="flowLegend" style="position: fixed; bottom: 50px; left: 10px; 
                  z-index: 9999; background-color: white; padding: 10px;
                  border-radius: 5px; box-shadow: 0 0 5px rgba(0,0,0,0.4);">
      <h4 style="margin-top: 0;">Flow Intensity</h4>
      <div style="display: flex; flex-direction: column; gap: 5px;">
        <div>
          <i style="background: linear-gradient(to right, #1a9850, #d73027); 
                    width: 120px; height: 10px; display: inline-block;"></i>
        </div>
        <div style="display: flex; justify-content: space-between; width: 120px;">
          <span style="font-size: 10px;">Low</span>
          <span style="font-size: 10px;">High</span>
        </div>
      </div>
      <div style="font-size: 10px; margin-top: 8px; color: #666; font-style: italic;">
        Drag me by clicking and holding
      </div>
    </div>
    
    <script>
      // Make the legend draggable
      var flowLegend = document.getElementById('flowLegend');
      var isDragging = false;
      var offsetX, offsetY;
      
      flowLegend.addEventListener('mousedown', function(e) {
        isDragging = true;
        offsetX = e.clientX - flowLegend.getBoundingClientRect().left;
        offsetY = e.clientY - flowLegend.getBoundingClientRect().top;
        flowLegend.style.cursor = 'grabbing';
      });
      
      document.addEventListener('mousemove', function(e) {
        if (isDragging) {
          flowLegend.style.left = (e.clientX - offsetX) + 'px';
          flowLegend.style.top = (e.clientY - offsetY) + 'px';
          flowLegend.style.right = 'auto';
          flowLegend.style.bottom = 'auto';
        }
      });
      
      document.addEventListener('mouseup', function() {
        isDragging = false;
        flowLegend.style.cursor = 'grab';
      });
      
      flowLegend.style.cursor = 'grab';
    </script>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # Add layer control
    folium.LayerControl(position='topright', collapsed=False).add_to(m)
    
    # Save map
    os.makedirs(output_dir, exist_ok=True)
    map_path = os.path.join(output_dir, "flow_map.html")
    m.save(map_path)
    print(f"Flow map saved to {map_path}")
    
    return m

def create_flow_visuals(trips_df, stations_df, station_mapping, output_dir="results"):
    """Create flow visualization maps for different time periods"""
    print("Generating flow maps and OD matrix visualizations...")
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Process each time period
    for period_name, (start_hour, end_hour) in TIME_PERIODS.items():
        # Filter trips for this time period
        if start_hour < end_hour:
            period_trips = trips_df[(trips_df['start_hour'] >= start_hour) & 
                                    (trips_df['start_hour'] < end_hour)]
        else:
            # Handle overnight periods (e.g., 10 PM - 5 AM)
            period_trips = trips_df[(trips_df['start_hour'] >= start_hour) | 
                                    (trips_df['start_hour'] < end_hour)]
        
        print(f"Processing {period_name} with {len(period_trips)} trips")
        
        # Compute flows between stations
        flows = []
        for _, trip in period_trips.iterrows():
            if trip['start_station_name'] in station_mapping and trip['end_station_name'] in station_mapping:
                flows.append((trip['start_station_name'], trip['end_station_name'], 1))
        
        # Aggregate flows
        flow_counts = {}
        for origin, dest, count in flows:
            key = (origin, dest)
            flow_counts[key] = flow_counts.get(key, 0) + count
        
        # Convert to list of tuples and sort by count
        aggregated_flows = [(origin, dest, count) 
                          for (origin, dest), count in flow_counts.items()]
        top_flows = sorted(aggregated_flows, key=lambda x: x[2], reverse=True)[:100]
        
        # Create a filtered flow map for this time period
        period_title = f"Bike Trip Flows: {period_name}"
        period_output_dir = os.path.join(output_dir, "time_periods")
        os.makedirs(period_output_dir, exist_ok=True)
        
        if top_flows:
            create_flow_map(
                stations_df,
                top_flows,
                output_dir=period_output_dir,
                title=period_title
            )
    
    # Create an overall flow map with all trips
    all_flows = []
    for _, trip in trips_df.iterrows():
        if trip['start_station_name'] in station_mapping and trip['end_station_name'] in station_mapping:
            all_flows.append((trip['start_station_name'], trip['end_station_name'], 1))
    
    # Aggregate all flows
    all_flow_counts = {}
    for origin, dest, count in all_flows:
        key = (origin, dest)
        all_flow_counts[key] = all_flow_counts.get(key, 0) + count
    
    # Convert to list of tuples and get top flows
    all_aggregated_flows = [(origin, dest, count) 
                          for (origin, dest), count in all_flow_counts.items()]
    top_all_flows = sorted(all_aggregated_flows, key=lambda x: x[2], reverse=True)[:100]
    
    # Create overall flow map
    create_flow_map(
        stations_df,
        top_all_flows,
        output_dir=output_dir,
        title="Bike Trip Flows: Overall"
    )
    
    print("Flow visualizations completed")
    return True

def main():
    """Main function to orchestrate the entire workflow"""
    try:
        # Set parameters from runtime config
        data_dir = RUNTIME_CONFIG['DATA_DIR']
        data_files_pattern = os.path.join(data_dir, RUNTIME_CONFIG['DATA_FILES_PATTERN'])
        output_dir = RUNTIME_CONFIG['OUTPUT_DIR']
        max_stations = RUNTIME_CONFIG['MAX_STATIONS']
        sample_rate = RUNTIME_CONFIG['SAMPLE_RATE']
        
        # Print analysis parameters
        print("Starting analysis with parameters:")
        print({
            'data_dir': data_dir,
            'output_dir': output_dir,
            'max_stations': max_stations,
            'sample_rate': sample_rate
        })
        
        # Create output directory
        os.makedirs(output_dir, exist_ok=True)
        
        # Load data
        trips_df = load_citibike_data(data_files_pattern, sample_rate=sample_rate)
        
        # Get top stations
        station_mapping, station_data, top_stations, stations_df = get_top_stations(trips_df, max_stations)
        
        print("\nGenerating visualizations...")
        
        # Create interactive station map
        interactive_station_map = create_interactive_station_map(
            stations_df,
            map_title="Citibike Stations",
            output_dir=output_dir
        )
        
        # Create flow visualizations
        create_flow_visuals(trips_df, stations_df, station_mapping, output_dir)
        
        print("\nAnalysis completed successfully!")
        return 0
        
    except Exception as e:
        print(f"\nError in main execution: {str(e)}")
        import traceback
        traceback.print_exc()
        return 1

if __name__ == "__main__":
    main()

Starting analysis with parameters:
{'data_dir': 'data/citibike', 'output_dir': 'results', 'max_stations': 50, 'sample_rate': 1}
Loading Citibike data with 100.0% sampling rate
Found 12 files to process
Processing file: JC202401citibiketripdata.csv
Extracted month: 1
Loaded 50661 sampled trips from JC202401citibiketripdata.csv
Processing file: JC202402citibiketripdata.csv
Extracted month: 2
Loaded 55613 sampled trips from JC202402citibiketripdata.csv
Processing file: JC202403citibiketripdata.csv
Extracted month: 3
Loaded 65581 sampled trips from JC202403citibiketripdata.csv
Processing file: JC202404citibiketripdata.csv
Extracted month: 4
Loaded 79116 sampled trips from JC202404citibiketripdata.csv
Processing file: JC202405citibiketripdata.csv
Extracted month: 5
Loaded 97479 sampled trips from JC202405citibiketripdata.csv
Processing file: JC202406citibiketripdata.csv
Extracted month: 6
Loaded 111115 sampled trips from JC202406citibiketripdata.csv
Processing file: JC202407citibiketripdata